# Approach3 Window Coverage Pre-analysis

## tl;dr

This diagnostic notebook scans temporal windows around one or more target dates and estimates how much of an AOI can be classified by the planned source hierarchy:

1. Dynamic World valid pixels.
2. OPERA DSWx-HLS Landsat valid pixels only where Dynamic World is invalid.
3. OPERA DSWx-S1 valid pixels only where Dynamic World and HLS are invalid.

The main outputs are CSV files saved under `notebooks/outputs/window_coverage_preanalysis/<run_label>/`:

- `window_scan_results.csv`: one row per target date and window size.
- `optimal_windows.csv`: the smallest window meeting the configured coverage target, or the best available window if the target is not met.
- `parameters.json`: run settings.
- `aoi.geojson`: the AOI used for the run.

Run the notebook for a small number of target dates first. Increase the number of targets only after runtime and AOI scale look reasonable.


## Context & Methods

### Key Assumptions

- `END_DATE` is exclusive, matching Earth Engine `filterDate(start, end)` semantics.
- Dynamic World is Sentinel-2-derived, so OPERA DSWx-HLS Sentinel-2/MSI is excluded by default. HLS is therefore treated as the Landsat gap-fill source.
- Coverage means **valid classification coverage**, not merely source image footprint coverage.
- Coverage is estimated at `AREA_SCALE_M`; use a coarse scale such as 300 m for exploratory scans, then lower it for final checks.
- A literal 100% coverage target can be too strict because product masks, edge pixels, clouds, layover/shadow, and footprint differences can leave small persistent gaps.

The notebook is intentionally simple: it scans `+- N` day windows around target dates and reports source contribution percentages. It does not export water products.


## 1. Parameters

Edit these before running. Start with `TARGET_DATE_MODE = "manual"` and one or two dates. Use `TARGET_DATE_MODE = "fixed_interval"` or `"dw_dates"` after the single-date scan behaves as expected.


In [1]:
# Earth Engine
EE_PROJECT = "hardy-tenure-383607"
USE_HIGH_VOLUME_ENDPOINT = False

# AOI options: "drawn", "basin", "bbox", "point_buffer", or "geojson".
AOI_LABEL = "drawn_test"
AOI_MODE = "drawn"
HYBAS_ID = 1041259950
HYDROBASINS_LEVEL = 4
AOI_BBOX = None  # Example: (29.0, -7.0, 30.0, -6.0)
AOI_GEOJSON_PATH = None
TEST_AOI_POINT_LON = 29.75
TEST_AOI_POINT_LAT = -6.5
TEST_AOI_BUFFER_M = 20000
MAP_CENTER = [-6.5, 29.5]  # [lat, lon]
MAP_ZOOM = 6

# Date range for candidate target dates. END_DATE is exclusive.
START_DATE = "2025-01-01"
END_DATE = "2025-07-01"

# Target dates to scan: "manual", "fixed_interval", or "dw_dates".
TARGET_DATE_MODE = "manual"
MANUAL_TARGET_DATES = ["2025-01-01"]
TARGET_INTERVAL_DAYS = 16
MAX_TARGET_DATES = 12

# Window scan settings.
MAX_WINDOW_DAYS = 30
WINDOW_STEP_DAYS = 1
COVERAGE_TARGET_PCT = 99.0
AREA_SCALE_M = 300
REDUCE_TILE_SCALE = 4

# Source settings.
INCLUDE_HLS_SENTINEL2 = False  # Keep False: Dynamic World already uses Sentinel-2.

# Output settings.
OUTPUT_RUN_LABEL = "notebooks/outputs/window_coverage_preanalysis/drawn_test_20260702_102042"  # None builds a timestamped label.


## 2. Setup

This cell imports the local Approach3 helpers and initializes Earth Engine.


In [2]:
from __future__ import annotations

from datetime import date, datetime, timedelta, timezone
from pathlib import Path
import json
import sys

import ee
import geemap
import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "Approach3":
    APPROACH3_ROOT = cwd
elif (cwd / "Approaches" / "Approach3").exists():
    APPROACH3_ROOT = cwd / "Approaches" / "Approach3"
else:
    APPROACH3_ROOT = next(parent for parent in cwd.parents if parent.name == "Approach3")

sys.path.insert(0, str(APPROACH3_ROOT / "src"))

from sw_dws1_approach3.aoi import AoiConfig, aoi_summary, basin_aoi, resolve_aoi
from sw_dws1_approach3.datasets import (
    DynamicWorldThresholds,
    OperaHlsWtrClass,
    OperaS1WtrClass,
    dynamic_world_collection,
    opera_dswx_hls_collection,
    opera_dswx_s1_collection,
)
from sw_dws1_approach3.gee_session import initialize_earth_engine
from sw_dws1_approach3.periods import validate_date_window

if EE_PROJECT == "your-google-cloud-project-id":
    raise ValueError("Set EE_PROJECT before running the notebook.")

validate_date_window(START_DATE, END_DATE)
initialize_earth_engine(project=EE_PROJECT, use_high_volume_endpoint=USE_HIGH_VOLUME_ENDPOINT)

run_label = OUTPUT_RUN_LABEL or f"{AOI_LABEL}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = APPROACH3_ROOT / "notebooks" / "outputs" / "window_coverage_preanalysis" / run_label
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Approach3 root:", APPROACH3_ROOT)
print("Output directory:", OUTPUT_DIR)


Approach3 root: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3
Output directory: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\window_coverage_preanalysis\notebooks\outputs\window_coverage_preanalysis\drawn_test_20260702_102042


## 3. AOI Selection

If `AOI_MODE = "drawn"`, draw one polygon or rectangle on this map, then run the next cell. For repeated latitude tests, rerun the notebook with a different `AOI_LABEL` and AOI.


In [3]:
aoi_config = AoiConfig(
    mode=AOI_MODE,
    hybas_id=HYBAS_ID,
    hydrobasins_level=HYDROBASINS_LEVEL,
    bbox=AOI_BBOX,
    point_lon=TEST_AOI_POINT_LON,
    point_lat=TEST_AOI_POINT_LAT,
    point_buffer_m=TEST_AOI_BUFFER_M,
    geojson_path=AOI_GEOJSON_PATH,
)

reference_aoi = basin_aoi(HYBAS_ID, level=HYDROBASINS_LEVEL) if AOI_MODE == "drawn" else resolve_aoi(aoi_config)

draw_map = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM, ee_initialize=False)
draw_map.addLayer(reference_aoi, {"color": "cyan"}, "Configured/reference AOI")
draw_map.addLayerControl()
draw_map


Map(center=[-6.5, 29.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## 4. Confirm AOI

This saves the AOI geometry used by the scan to `aoi.geojson` inside the output run folder.


In [4]:
drawn_geometry = None
if AOI_MODE == "drawn":
    if draw_map.draw_last_feature is None:
        raise ValueError("AOI_MODE is 'drawn', but no geometry has been drawn on the map above.")
    drawn_geometry = draw_map.draw_last_feature.geometry()

aoi = resolve_aoi(aoi_config, custom_geometry=drawn_geometry)
aoi_info = aoi_summary(aoi).getInfo()
aoi_area_km2 = float(aoi_info["area_km2"])

feature_geojson = {
    "type": "Feature",
    "properties": {"aoi_label": AOI_LABEL, "aoi_mode": AOI_MODE},
    "geometry": aoi.getInfo(),
}
with (OUTPUT_DIR / "aoi.geojson").open("w", encoding="utf-8") as f:
    json.dump(feature_geojson, f, indent=2)

print("AOI summary:", aoi_info)
print(f"AOI area km2: {aoi_area_km2:,.2f}")


AOI summary: {'area_km2': 1292637.807833093, 'bounds': [[[24.916992, -10.43852], [35.244141, -10.43852], [35.244141, -0.257491], [24.916992, -0.257491], [24.916992, -10.43852]]]}
AOI area km2: 1,292,637.81


## 5. Target Dates

Target dates can be supplied manually, generated at a fixed interval, or sampled from unique Dynamic World acquisition dates in the range.


In [5]:
def _parse_date(value: str) -> date:
    return date.fromisoformat(value)


def _date_to_text(value: date) -> str:
    return value.isoformat()


def fixed_interval_dates(start_date: str, end_date: str, interval_days: int, max_dates: int) -> list[str]:
    start = _parse_date(start_date)
    end = _parse_date(end_date)
    out = []
    current = start
    while current < end and len(out) < max_dates:
        out.append(_date_to_text(current))
        current += timedelta(days=interval_days)
    return out


def dynamic_world_dates(start_date: str, end_date: str, max_dates: int) -> list[str]:
    times = dynamic_world_collection(aoi, start_date, end_date).aggregate_array("system:time_start").getInfo()
    dates = sorted({datetime.fromtimestamp(t / 1000, tz=timezone.utc).date().isoformat() for t in times})
    return dates[:max_dates]


if TARGET_DATE_MODE == "manual":
    target_dates = MANUAL_TARGET_DATES[:MAX_TARGET_DATES]
elif TARGET_DATE_MODE == "fixed_interval":
    target_dates = fixed_interval_dates(START_DATE, END_DATE, TARGET_INTERVAL_DAYS, MAX_TARGET_DATES)
elif TARGET_DATE_MODE == "dw_dates":
    target_dates = dynamic_world_dates(START_DATE, END_DATE, MAX_TARGET_DATES)
else:
    raise ValueError('TARGET_DATE_MODE must be "manual", "fixed_interval", or "dw_dates".')

if not target_dates:
    raise ValueError("No target dates selected.")

target_df = pd.DataFrame({"target_date": target_dates})
display(target_df)


,target_date
0,2025-01-01


## 6. Coverage Helpers

The valid masks below follow the current Approach3 source logic:

- Dynamic World valid if open water, valid non-water, or flooded vegetation is confidently identified.
- HLS Landsat valid if the OPERA HLS WTR class is below snow/ice/cloud/ocean classes.
- S1 valid if the OPERA S1 WTR class is below hand/layover/ocean mask classes.


In [6]:
thresholds = DynamicWorldThresholds()


def dw_valid_observation(image: ee.Image) -> ee.Image:
    image = ee.Image(image)
    water = image.select("water")
    flooded = image.select("flooded_vegetation")
    valid = water.gt(thresholds.water).Or(water.lte(thresholds.nonwater)).Or(
        flooded.gt(thresholds.flooded_vegetation)
    )
    return valid.rename("valid").toByte().updateMask(water.mask())


def hls_valid_observation(image: ee.Image) -> ee.Image:
    wtr = ee.Image(image).select("WTR_Water_classification")
    return wtr.lt(OperaHlsWtrClass.SNOW_ICE).rename("valid").toByte().updateMask(wtr.mask())


def s1_valid_observation(image: ee.Image) -> ee.Image:
    wtr = ee.Image(image).select("WTR_Water_classification")
    return wtr.lt(OperaS1WtrClass.HAND_MASKED).rename("valid").toByte().updateMask(wtr.mask())


def empty_valid_image() -> ee.Image:
    return ee.Image.constant(0).rename("valid").toByte().clip(aoi)


def any_valid_mask(collection: ee.ImageCollection, valid_fn) -> ee.Image:
    valid_collection = ee.ImageCollection(
        collection.map(lambda image: valid_fn(ee.Image(image)).unmask(0))
    ).merge(ee.ImageCollection([empty_valid_image()]))
    return valid_collection.sum().gt(0).rename("valid_any").clip(aoi)


def window_bounds(target_date: str, window_days: int) -> tuple[str, str]:
    target = _parse_date(target_date)
    start = target - timedelta(days=window_days)
    end = target + timedelta(days=window_days + 1)
    return start.isoformat(), end.isoformat()


def build_window_sources(target_date: str, window_days: int) -> dict:
    start, end = window_bounds(target_date, window_days)
    dw_collection = dynamic_world_collection(aoi, start, end)
    hls_collection = opera_dswx_hls_collection(
        aoi,
        start,
        end,
        include_sentinel2=INCLUDE_HLS_SENTINEL2,
    )
    s1_collection = opera_dswx_s1_collection(aoi, start, end)

    dw_valid = any_valid_mask(dw_collection, dw_valid_observation).unmask(0).eq(1)
    hls_valid = any_valid_mask(hls_collection, hls_valid_observation).unmask(0).eq(1)
    s1_valid = any_valid_mask(s1_collection, s1_valid_observation).unmask(0).eq(1)

    dw_gap = dw_valid.Not()
    hls_used = dw_gap.And(hls_valid)
    after_hls_gap = dw_gap.And(hls_valid.Not())
    s1_used = after_hls_gap.And(s1_valid)
    final_valid = dw_valid.Or(hls_used).Or(s1_used)
    remaining_gap = final_valid.Not()

    return {
        "target_date": target_date,
        "window_days": window_days,
        "window_start": start,
        "window_end_exclusive": end,
        "collections": {"dw": dw_collection, "hls": hls_collection, "s1": s1_collection},
        "masks": {
            "dw_valid": dw_valid,
            "hls_valid": hls_valid,
            "s1_valid": s1_valid,
            "dw_gap": dw_gap,
            "hls_used": hls_used,
            "after_hls_gap": after_hls_gap,
            "s1_used": s1_used,
            "final_valid": final_valid,
            "remaining_gap": remaining_gap,
        },
    }


def area_stats(mask_dict: dict[str, ee.Image]) -> dict[str, float]:
    area_bands = []
    for name, mask in mask_dict.items():
        area_bands.append(ee.Image.pixelArea().rename(name).updateMask(mask))
    stats = ee.Image.cat(area_bands).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=AREA_SCALE_M,
        maxPixels=1e13,
        tileScale=REDUCE_TILE_SCALE,
    ).getInfo()
    return {name: (stats.get(name) or 0) / 1_000_000 for name in mask_dict}


def pct(area_km2: float, denominator_km2: float) -> float:
    if denominator_km2 <= 0:
        return 0.0
    return 100.0 * area_km2 / denominator_km2


def scan_target_window(target_date: str, window_days: int) -> dict:
    sources = build_window_sources(target_date, window_days)
    masks = sources["masks"]
    areas = area_stats(masks)
    counts = ee.Dictionary({
        "dw_image_count": sources["collections"]["dw"].size(),
        "hls_image_count": sources["collections"]["hls"].size(),
        "s1_image_count": sources["collections"]["s1"].size(),
    }).getInfo()

    dw_gap_area = areas["dw_gap"]
    after_hls_gap_area = areas["after_hls_gap"]

    return {
        "aoi_label": AOI_LABEL,
        "target_date": target_date,
        "window_days": window_days,
        "window_start": sources["window_start"],
        "window_end_exclusive": sources["window_end_exclusive"],
        **counts,
        "dw_valid_pct": pct(areas["dw_valid"], aoi_area_km2),
        "hls_valid_pct": pct(areas["hls_valid"], aoi_area_km2),
        "s1_valid_pct": pct(areas["s1_valid"], aoi_area_km2),
        "hls_used_pct_aoi": pct(areas["hls_used"], aoi_area_km2),
        "s1_used_pct_aoi": pct(areas["s1_used"], aoi_area_km2),
        "final_valid_pct": pct(areas["final_valid"], aoi_area_km2),
        "remaining_gap_pct": pct(areas["remaining_gap"], aoi_area_km2),
        "hls_fills_dw_gap_pct": pct(areas["hls_used"], dw_gap_area),
        "s1_fills_after_hls_gap_pct": pct(areas["s1_used"], after_hls_gap_area),
        "dw_valid_km2": areas["dw_valid"],
        "hls_used_km2": areas["hls_used"],
        "s1_used_km2": areas["s1_used"],
        "final_valid_km2": areas["final_valid"],
        "remaining_gap_km2": areas["remaining_gap"],
    }


## 7. Run Window Scan

This is the main live Earth Engine step. Keep `MAX_TARGET_DATES` and `MAX_WINDOW_DAYS` modest until you know runtime for the AOI.


In [7]:
window_days_list = list(range(0, MAX_WINDOW_DAYS + 1, WINDOW_STEP_DAYS))
print(f"Scanning {len(target_dates)} target dates x {len(window_days_list)} windows = {len(target_dates) * len(window_days_list)} rows")

rows = []
for target_date in target_dates:
    for window_days in window_days_list:
        print(f"Scanning target={target_date}, window=+-{window_days} days")
        rows.append(scan_target_window(target_date, window_days))

window_df = pd.DataFrame(rows)
window_csv = OUTPUT_DIR / "window_scan_results.csv"
window_df.to_csv(window_csv, index=False)

print("Saved:", window_csv)
display(window_df.head(20))


Scanning 1 target dates x 31 windows = 31 rows
Scanning target=2025-01-01, window=+-0 days
Scanning target=2025-01-01, window=+-1 days
Scanning target=2025-01-01, window=+-2 days
Scanning target=2025-01-01, window=+-3 days
Scanning target=2025-01-01, window=+-4 days
Scanning target=2025-01-01, window=+-5 days
Scanning target=2025-01-01, window=+-6 days
Scanning target=2025-01-01, window=+-7 days
Scanning target=2025-01-01, window=+-8 days
Scanning target=2025-01-01, window=+-9 days
Scanning target=2025-01-01, window=+-10 days
Scanning target=2025-01-01, window=+-11 days
Scanning target=2025-01-01, window=+-12 days
Scanning target=2025-01-01, window=+-13 days
Scanning target=2025-01-01, window=+-14 days
Scanning target=2025-01-01, window=+-15 days
Scanning target=2025-01-01, window=+-16 days
Scanning target=2025-01-01, window=+-17 days
Scanning target=2025-01-01, window=+-18 days
Scanning target=2025-01-01, window=+-19 days
Scanning target=2025-01-01, window=+-20 days
Scanning target=20

,aoi_label,target_date,window_days,window_start,window_end_exclusive,dw_image_count,hls_image_count,s1_image_count,dw_valid_pct,hls_valid_pct,...,s1_used_pct_aoi,final_valid_pct,remaining_gap_pct,hls_fills_dw_gap_pct,s1_fills_after_hls_gap_pct,dw_valid_km2,hls_used_km2,s1_used_km2,final_valid_km2,remaining_gap_km2
0,drawn_test,2025-01-01,0,2025-01-01,2025-01-02,6,56,62,0.181649,2.319435,...,22.785378,25.132713,74.436516,2.179031,23.436468,2348.069130,27994.476439,294532.406408,3.248750e+05,962194.551949
1,drawn_test,2025-01-01,1,2024-12-31,2025-01-03,25,114,62,4.984304,8.068677,...,21.961423,30.325335,69.243894,3.573094,24.079103,64429.003668,43686.092838,283881.651712,3.919967e+05,895072.755709
2,drawn_test,2025-01-01,2,2024-12-30,2025-01-04,58,241,162,11.767517,20.834685,...,48.349780,70.275450,29.293779,11.569425,62.271463,152111.367327,131308.130136,624987.539806,9.084070e+05,378662.466657
3,drawn_test,2025-01-01,3,2024-12-29,2025-01-05,93,348,234,18.276281,32.738118,...,47.726905,81.623798,17.945431,19.215211,72.674292,236246.117948,201917.934690,616936.022410,1.055100e+06,231969.428878
4,drawn_test,2025-01-01,4,2024-12-28,2025-01-06,133,437,236,23.555024,38.335854,...,44.928908,84.572316,14.996914,21.164969,74.974204,304481.149775,207964.522022,580768.054739,1.093214e+06,193855.777390
5,drawn_test,2025-01-01,5,2024-12-27,2025-01-07,187,537,335,31.006059,47.388472,...,50.125104,99.383021,0.186208,26.620499,99.629888,400796.038293,235930.420020,647936.051075,1.284663e+06,2406.994539
6,drawn_test,2025-01-01,6,2024-12-26,2025-01-08,239,633,335,33.609987,54.502959,...,43.134842,99.403538,0.165691,34.352592,99.617346,434455.393568,292895.044120,557577.277745,1.284928e+06,2141.788494
7,drawn_test,2025-01-01,7,2024-12-25,2025-01-09,272,731,434,34.793983,58.055685,...,39.686647,99.409070,0.160160,38.484515,99.598061,449760.182342,322234.432309,513004.604546,1.284999e+06,2070.284730
8,drawn_test,2025-01-01,8,2024-12-24,2025-01-10,320,828,436,36.951491,63.463735,...,34.265329,99.420745,0.148485,45.041430,99.568532,477648.937372,364574.596846,442926.601098,1.285150e+06,1919.368610
9,drawn_test,2025-01-01,9,2024-12-23,2025-01-11,367,920,508,39.210166,65.297571,...,32.585323,99.427823,0.141406,45.779926,99.567918,506845.429626,357186.000682,421210.204251,1.285242e+06,1827.869367


## 8. Select Minimum Window Per Target Date

The selected window is the smallest one meeting `COVERAGE_TARGET_PCT`. If none meets the target, the notebook keeps the best available window and marks `meets_target = False`.


In [8]:
if "window_df" not in globals():
    if "OUTPUT_DIR" in globals() and (OUTPUT_DIR / "window_scan_results.csv").exists():
        window_df = pd.read_csv(OUTPUT_DIR / "window_scan_results.csv")
        print("Loaded:", OUTPUT_DIR / "window_scan_results.csv")
    else:
        raise RuntimeError(
            "window_df is not available. Run Sections 1-7 first, or make sure "
            "window_scan_results.csv exists in OUTPUT_DIR."
        )


def select_optimal_window(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values(["window_days", "remaining_gap_pct"])
    eligible = ordered[ordered["final_valid_pct"] >= COVERAGE_TARGET_PCT]
    if not eligible.empty:
        row = eligible.iloc[0].copy()
        row["meets_target"] = True
        return row

    best = group.sort_values(["final_valid_pct", "window_days"], ascending=[False, True]).iloc[0].copy()
    best["meets_target"] = False
    return best

optimal_rows = []
for target_date, group in window_df.groupby("target_date", sort=True):
    row = select_optimal_window(group).copy()
    row["target_date"] = target_date
    optimal_rows.append(row)
optimal_df = pd.DataFrame(optimal_rows).reset_index(drop=True)

# Keep target_date as a normal column even if pandas/groupby behavior changes.
if "target_date" not in optimal_df.columns:
    optimal_df = optimal_df.reset_index()
if "target_date" not in optimal_df.columns:
    raise RuntimeError(f"Could not build optimal_df with target_date column. Columns: {list(optimal_df.columns)}")

optimal_csv = OUTPUT_DIR / "optimal_windows.csv"
optimal_df.to_csv(optimal_csv, index=False)

print("Coverage target pct:", COVERAGE_TARGET_PCT)
print("Saved:", optimal_csv)
display(optimal_df)


Coverage target pct: 99.0
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\window_coverage_preanalysis\notebooks\outputs\window_coverage_preanalysis\drawn_test_20260702_102042\optimal_windows.csv


,aoi_label,target_date,window_days,window_start,window_end_exclusive,dw_image_count,hls_image_count,s1_image_count,dw_valid_pct,hls_valid_pct,...,final_valid_pct,remaining_gap_pct,hls_fills_dw_gap_pct,s1_fills_after_hls_gap_pct,dw_valid_km2,hls_used_km2,s1_used_km2,final_valid_km2,remaining_gap_km2,meets_target
0,drawn_test,2025-01-01,5,2024-12-27,2025-01-07,187,537,335,31.006059,47.388472,...,99.383021,0.186208,26.620499,99.629888,400796.038293,235930.42002,647936.051075,1.284663e+06,2406.994539,True


## 9. Quick Coverage Summary

This cell avoids matplotlib because plotting backends can crash the notebook kernel in some local environments. It displays a compact table and simple inline SVG coverage curves instead.

Color meaning in the coverage curves:

- Blue: Dynamic World valid coverage.
- Orange: OPERA HLS Landsat pixels used to fill Dynamic World gaps.
- Purple: OPERA S1 pixels used to fill gaps remaining after Dynamic World and HLS.
- Black: final valid coverage after applying the hierarchy.
- Gray dashed line: configured coverage target.


In [9]:
from IPython.display import HTML, display

if "window_df" not in globals():
    if "OUTPUT_DIR" in globals() and (OUTPUT_DIR / "window_scan_results.csv").exists():
        window_df = pd.read_csv(OUTPUT_DIR / "window_scan_results.csv")
        print("Loaded:", OUTPUT_DIR / "window_scan_results.csv")
    else:
        raise RuntimeError(
            "window_df is not available. Run Sections 1-7 first, or make sure "
            "window_scan_results.csv exists in OUTPUT_DIR."
        )

summary_cols = [
    "target_date",
    "window_days",
    "dw_valid_pct",
    "hls_used_pct_aoi",
    "s1_used_pct_aoi",
    "final_valid_pct",
    "remaining_gap_pct",
]
summary_table = window_df[summary_cols].copy()
display(summary_table.head(40))


def _svg_polyline(points: list[tuple[float, float]], color: str, width: int = 2) -> str:
    if not points:
        return ""
    point_text = " ".join(f"{x:.1f},{y:.1f}" for x, y in points)
    return f'<polyline points="{point_text}" fill="none" stroke="{color}" stroke-width="{width}" />'


def _coverage_svg(group: pd.DataFrame, target_date: str) -> str:
    metrics = [
        ("dw_valid_pct", "DW valid", "#419bdf"),
        ("hls_used_pct_aoi", "HLS used", "#e49635"),
        ("s1_used_pct_aoi", "S1 used", "#7a87c6"),
        ("final_valid_pct", "Final valid", "#111111"),
    ]
    width, height = 760, 300
    left, right, top, bottom = 55, 20, 25, 45
    plot_w = width - left - right
    plot_h = height - top - bottom
    x_min = float(group["window_days"].min())
    x_max = float(group["window_days"].max())
    if x_max == x_min:
        x_max = x_min + 1

    def sx(value: float) -> float:
        return left + (float(value) - x_min) / (x_max - x_min) * plot_w

    def sy(value: float) -> float:
        return top + (100.0 - float(value)) / 100.0 * plot_h

    lines = []
    for metric, _label, color in metrics:
        points = [(sx(row["window_days"]), sy(row[metric])) for _, row in group.sort_values("window_days").iterrows()]
        lines.append(_svg_polyline(points, color))

    target_y = sy(COVERAGE_TARGET_PCT)
    legend = "".join(
        f'<span style="display:inline-block;margin-right:14px;"><span style="display:inline-block;width:12px;height:12px;background:{color};margin-right:4px;"></span>{label}</span>'
        for _, label, color in metrics
    )

    return f'''
    <div style="margin:16px 0 22px 0; font-family:Arial, sans-serif;">
      <div style="font-weight:600; margin-bottom:4px;">Coverage scan: {target_date}</div>
      <svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img">
        <rect x="0" y="0" width="{width}" height="{height}" fill="white" />
        <line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#999" />
        <line x1="{left}" y1="{top + plot_h}" x2="{left + plot_w}" y2="{top + plot_h}" stroke="#999" />
        <line x1="{left}" y1="{target_y}" x2="{left + plot_w}" y2="{target_y}" stroke="#666" stroke-dasharray="5,4" />
        <text x="8" y="{sy(100) + 4:.1f}" font-size="11" fill="#555">100%</text>
        <text x="14" y="{sy(50) + 4:.1f}" font-size="11" fill="#555">50%</text>
        <text x="20" y="{sy(0) + 4:.1f}" font-size="11" fill="#555">0%</text>
        <text x="{left}" y="{height - 10}" font-size="11" fill="#555">+- {x_min:g} days</text>
        <text x="{left + plot_w - 55}" y="{height - 10}" font-size="11" fill="#555">+- {x_max:g} days</text>
        <text x="{left + plot_w - 135}" y="{target_y - 5:.1f}" font-size="11" fill="#555">target {COVERAGE_TARGET_PCT:g}%</text>
        {''.join(lines)}
      </svg>
      <div style="font-size:12px;">{legend}</div>
    </div>
    '''

html_parts = []
for target_date, group in window_df.groupby("target_date"):
    html_parts.append(_coverage_svg(group, str(target_date)))

display(HTML("".join(html_parts)))


,target_date,window_days,dw_valid_pct,hls_used_pct_aoi,s1_used_pct_aoi,final_valid_pct,remaining_gap_pct
0,2025-01-01,0,0.181649,2.165686,22.785378,25.132713,74.436516
1,2025-01-01,1,4.984304,3.379608,21.961423,30.325335,69.243894
2,2025-01-01,2,11.767517,10.158153,48.349780,70.275450,29.293779
3,2025-01-01,3,18.276281,15.620612,47.726905,81.623798,17.945431
4,2025-01-01,4,23.555024,16.088383,44.928908,84.572316,14.996914
5,2025-01-01,5,31.006059,18.251858,50.125104,99.383021,0.186208
6,2025-01-01,6,33.609987,22.658709,43.134842,99.403538,0.165691
7,2025-01-01,7,34.793983,24.928439,39.686647,99.409070,0.160160
8,2025-01-01,8,36.951491,28.203925,34.265329,99.420745,0.148485
9,2025-01-01,9,39.210166,27.632334,32.585323,99.427823,0.141406


## 10. Optional Diagnostic Map

This map visualizes one selected target date/window. It shows which source contributes to final valid coverage. This is the first practical version of a regional map of source contribution within the AOI.

Color meaning in the diagnostic map:

- Source contribution layer: blue = Dynamic World, orange = OPERA HLS Landsat, purple = OPERA S1.
- Remaining gap layer: red = pixels still not covered by any source in the selected window.
- Date-distance layers: green = closer to the target date, yellow = intermediate distance, red = farther from the target date.
- Max source date difference layer: green = small temporal mismatch between available sources, yellow = intermediate mismatch, red = larger mismatch.


In [10]:
from IPython.display import HTML, display


def _load_window_df_for_map() -> pd.DataFrame:
    if "window_df" in globals() and "target_date" in window_df.columns:
        return window_df
    if "OUTPUT_DIR" in globals() and (OUTPUT_DIR / "window_scan_results.csv").exists():
        loaded = pd.read_csv(OUTPUT_DIR / "window_scan_results.csv")
        print("Loaded:", OUTPUT_DIR / "window_scan_results.csv")
        return loaded
    raise RuntimeError(
        "window_df is not available. Run Sections 1-7 first, or make sure "
        "window_scan_results.csv exists in OUTPUT_DIR."
    )


def _select_optimal_window_for_map(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values(["window_days", "remaining_gap_pct"])
    eligible = ordered[ordered["final_valid_pct"] >= COVERAGE_TARGET_PCT]
    if not eligible.empty:
        row = eligible.iloc[0].copy()
        row["meets_target"] = True
        return row
    best = group.sort_values(["final_valid_pct", "window_days"], ascending=[False, True]).iloc[0].copy()
    best["meets_target"] = False
    return best


def _rebuild_optimal_df_for_map(source_window_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for target_date, group in source_window_df.groupby("target_date", sort=True):
        row = _select_optimal_window_for_map(group).copy()
        row["target_date"] = target_date
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


def _normalize_optimal_df_for_map() -> pd.DataFrame:
    if "optimal_df" in globals():
        candidate = optimal_df.copy()
        if "target_date" in candidate.columns:
            return candidate.reset_index(drop=True)
        if candidate.index.name == "target_date":
            candidate = candidate.reset_index()
            if "target_date" in candidate.columns:
                return candidate.reset_index(drop=True)
        print("Existing optimal_df is missing target_date; rebuilding from window scan results.")

    if "OUTPUT_DIR" in globals() and (OUTPUT_DIR / "optimal_windows.csv").exists():
        candidate = pd.read_csv(OUTPUT_DIR / "optimal_windows.csv")
        if "target_date" in candidate.columns:
            print("Loaded:", OUTPUT_DIR / "optimal_windows.csv")
            return candidate.reset_index(drop=True)
        print("Saved optimal_windows.csv is missing target_date; rebuilding from window scan results.")

    source_window_df = _load_window_df_for_map()
    rebuilt = _rebuild_optimal_df_for_map(source_window_df)
    rebuilt.to_csv(OUTPUT_DIR / "optimal_windows.csv", index=False)
    print("Rebuilt and saved:", OUTPUT_DIR / "optimal_windows.csv")
    return rebuilt


optimal_df = _normalize_optimal_df_for_map()
if "target_date" not in optimal_df.columns:
    raise RuntimeError(f"optimal_df still has no target_date column. Columns: {list(optimal_df.columns)}")

MAP_ROW_INDEX = 0
map_row = optimal_df.iloc[MAP_ROW_INDEX]
MAP_TARGET_DATE = str(map_row["target_date"])
MAP_WINDOW_DAYS = int(map_row["window_days"])

map_sources = build_window_sources(MAP_TARGET_DATE, MAP_WINDOW_DAYS)
map_masks = map_sources["masks"]

source_contribution = (
    ee.Image.constant(0)
    .where(map_masks["dw_valid"], 1)
    .where(map_masks["hls_used"], 2)
    .where(map_masks["s1_used"], 3)
    .updateMask(map_masks["final_valid"])
    .rename("source_contribution")
)


def add_closest_delta_bands(image: ee.Image, valid_fn, target_date: str) -> ee.Image:
    image = ee.Image(image)
    valid = valid_fn(image).unmask(0).eq(1)
    target = ee.Date(target_date)
    signed_delta = ee.Date(image.get("system:time_start")).difference(target, "day")
    abs_delta = signed_delta.abs()
    quality = ee.Image.constant(abs_delta.multiply(-1)).rename("closest_quality").toFloat().updateMask(valid)
    return image.addBands(ee.Image.cat([
        quality,
        ee.Image.constant(abs_delta).rename("abs_delta_days").toFloat().updateMask(valid),
        ee.Image.constant(signed_delta).rename("signed_delta_days").toFloat().updateMask(valid),
    ]))


def closest_delta_image(collection: ee.ImageCollection, valid_fn, target_date: str) -> ee.Image:
    masked_zero = ee.Image.constant(0).updateMask(ee.Image.constant(0))
    template = ee.Image.cat([
        ee.Image.constant(-1e9).rename("closest_quality").toFloat(),
        masked_zero.rename("abs_delta_days").toFloat(),
        masked_zero.rename("signed_delta_days").toFloat(),
    ])
    enriched = ee.ImageCollection(
        collection.map(lambda image: add_closest_delta_bands(ee.Image(image), valid_fn, target_date))
    ).merge(ee.ImageCollection([template]))
    return enriched.qualityMosaic("closest_quality").clip(aoi)


dw_delta = closest_delta_image(map_sources["collections"]["dw"], dw_valid_observation, MAP_TARGET_DATE)
hls_delta = closest_delta_image(map_sources["collections"]["hls"], hls_valid_observation, MAP_TARGET_DATE)
s1_delta = closest_delta_image(map_sources["collections"]["s1"], s1_valid_observation, MAP_TARGET_DATE)

max_pair_delta = ee.Image.cat([
    dw_delta.select("signed_delta_days").subtract(hls_delta.select("signed_delta_days")).abs()
        .updateMask(map_masks["dw_valid"].And(map_masks["hls_valid"])),
    dw_delta.select("signed_delta_days").subtract(s1_delta.select("signed_delta_days")).abs()
        .updateMask(map_masks["dw_valid"].And(map_masks["s1_valid"])),
    hls_delta.select("signed_delta_days").subtract(s1_delta.select("signed_delta_days")).abs()
        .updateMask(map_masks["hls_valid"].And(map_masks["s1_valid"])),
]).reduce(ee.Reducer.max()).rename("max_pair_delta_days")

qa_map = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM, ee_initialize=False)
qa_map.addLayer(aoi, {"color": "cyan"}, "AOI")
qa_map.addLayer(
    source_contribution,
    {"min": 1, "max": 3, "palette": ["419bdf", "e49635", "7a87c6"]},
    "Source contribution: 1=DW, 2=HLS, 3=S1",
)
qa_map.addLayer(map_masks["remaining_gap"].selfMask(), {"palette": ["red"]}, "Remaining gap", False)
qa_map.addLayer(dw_delta.select("abs_delta_days"), {"min": 0, "max": MAP_WINDOW_DAYS, "palette": ["1a9850", "fee08b", "d73027"]}, "DW closest date distance days", False)
qa_map.addLayer(hls_delta.select("abs_delta_days"), {"min": 0, "max": MAP_WINDOW_DAYS, "palette": ["1a9850", "fee08b", "d73027"]}, "HLS closest date distance days", False)
qa_map.addLayer(s1_delta.select("abs_delta_days"), {"min": 0, "max": MAP_WINDOW_DAYS, "palette": ["1a9850", "fee08b", "d73027"]}, "S1 closest date distance days", False)
qa_map.addLayer(max_pair_delta, {"min": 0, "max": MAP_WINDOW_DAYS * 2, "palette": ["1a9850", "fee08b", "d73027"]}, "Max source date difference days", False)
qa_map.addLayerControl()
print(f"Diagnostic map target={MAP_TARGET_DATE}, window=+-{MAP_WINDOW_DAYS} days")

map_legend_html = """
<div style="font-family:Arial, sans-serif; font-size:13px; line-height:1.45; margin:8px 0 12px 0;">
  <div style="font-weight:600; margin-bottom:4px;">Diagnostic map color legend</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:#419bdf;margin-right:6px;"></span>Dynamic World contribution</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:#e49635;margin-right:6px;"></span>OPERA HLS Landsat contribution</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:#7a87c6;margin-right:6px;"></span>OPERA S1 contribution</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:red;margin-right:6px;"></span>Remaining gap</div>
  <div style="margin-top:4px;"><span style="display:inline-block;width:52px;height:10px;background:linear-gradient(90deg,#1a9850,#fee08b,#d73027);margin-right:6px;"></span>Date-distance layers: green = closer, yellow = intermediate, red = farther</div>
</div>
"""
display(HTML(map_legend_html))
qa_map


Diagnostic map target=2025-01-01, window=+-5 days


Map(center=[-6.5, 29.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## 11. Save Parameters

The parameter file makes repeated latitude/AOI runs easier to compare.


In [ ]:
parameters = {
    "aoi_label": AOI_LABEL,
    "aoi_mode": AOI_MODE,
    "hybas_id": HYBAS_ID,
    "hydrobasins_level": HYDROBASINS_LEVEL,
    "start_date": START_DATE,
    "end_date_exclusive": END_DATE,
    "target_date_mode": TARGET_DATE_MODE,
    "manual_target_dates": MANUAL_TARGET_DATES,
    "target_interval_days": TARGET_INTERVAL_DAYS,
    "max_target_dates": MAX_TARGET_DATES,
    "max_window_days": MAX_WINDOW_DAYS,
    "window_step_days": WINDOW_STEP_DAYS,
    "coverage_target_pct": COVERAGE_TARGET_PCT,
    "area_scale_m": AREA_SCALE_M,
    "include_hls_sentinel2": INCLUDE_HLS_SENTINEL2,
    "output_dir": str(OUTPUT_DIR),
}
with (OUTPUT_DIR / "parameters.json").open("w", encoding="utf-8") as f:
    json.dump(parameters, f, indent=2)

print("Saved:", OUTPUT_DIR / "parameters.json")


## Takeaways

Fill this section after running the notebook. Suggested checks:

- Does the minimum window differ substantially between target dates?
- Does HLS Landsat reduce DW gaps enough, or does S1 dominate gap filling?
- Are windows longer near equatorial/cloudy AOIs than in higher-latitude AOIs?
- Does the selected window become too large for an “individual date” interpretation?

For latitude sensitivity, rerun the notebook with a few representative AOIs and distinct `AOI_LABEL` values, then compare the `optimal_windows.csv` files. Avoid a global scan until the AOI-level workflow and thresholds are stable.
